In [2]:
import time
import struct

import chipwhisperer as cw

In [3]:
scope = cw.scope()

In [4]:
scope.default_setup()
scope.io.tio1 = 'serial_tx'
scope.io.tio2 = 'serial_rx'
scope.io.tio3 = 'gpio_low'
scope.io.glitch_trig_mcx = 'glitch'

scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 131124                    to 5000                     
scope.clock.clkgen_freq                  changed from 0                         to 7363636.363636363        
scope.clock.adc_freq                     changed from 0                         to 29454545.454545453       
scope.clock.adc_rate                     changed from 0.0                       to 29454545.454545453       
scope.io.tio1                            changed from serial_tx                 to serial_rx                
scope.io.tio2                            changed from serial_rx                 to serial_tx                
scope.io.hs2       

In [5]:
scope.clock.clkgen_src = 'system'
scope.clock.clkgen_freq = 100e6
scope.clock.adc_mul = 1

target = cw.target(scope)
target.baud = 9600
target.flush()


scope.trigger.triggers = 'tio1'
scope.trigger.module = 'UART'
scope.gain.db = 12

# scope.trigger.triggers = 'tio3'
# scope.trigger.module = 'basic'


scope.UARTTrigger.enabled = True
scope.UARTTrigger.baud = 9600
# scope.UARTTrigger.set_pattern_match(0, '\x22\x03\x01\x00\x01\x00\xff\x03') # Synchronize
scope.UARTTrigger.set_pattern_match(0, '\xff\xff\xff\xff\xff\xff\xcf\x03') # ID Code
scope.UARTTrigger.trigger_source = 0
scope.UARTTrigger.rules_enabled = [0]


scope.glitch.enabled = True
scope.glitch.clk_src = "pll"

scope.io.glitch_hp = False
scope.io.glitch_lp = True
scope.io.glitch_trig_mcx = 'glitch'

scope.glitch.output = "enable_only"
scope.glitch.trigger_src = "ext_single" # glitch only after scope.arm() called

(ChipWhisperer Scope WARNING|File ChipWhispererHuskyClock.py:524) ADC frequency must be between 1MHz and 300000000.0MHz - ADC mul has been adjusted to 3
(ChipWhisperer Scope WARNING|File ChipWhispererHuskyClock.py:527) ADC frequency exceeds specification (200 MHz). 
            This may or may not work, depending on temperature, voltage, and luck.
            It may not work reliably.
            You can run scope.adc_test() to check whether ADC data is sampled properly by the FPGA,
            but this doesn't fully verify that the ADC is working properly.
            Set scope.clock.pll._no_warning_freq if you don't want
            to see this message anymore.
            
(ChipWhisperer Scope WARNING|File TraceWhisperer.py:1370) Coudln't achieve exact desired frequency (198513084.411621); setting to 200004112.243652 instead.


In [6]:
# gc = cw.GlitchController(groups=["success", "reset", "normal"], parameters=["ext_offset", "repeat"])
# min_delay = int(0e-3 * scope.clock.clkgen_freq)
# max_delay = int(35e-3 * scope.clock.clkgen_freq)

# print(min_delay, max_delay)

# N = 200
# step = (max_delay - min_delay) // N

# gc.set_range("ext_offset", min_delay, max_delay)
# gc.set_range("repeat", 5, 50)

# gc.set_global_step([step, 5])
# # gc.set_step("ext_offset", step)
# # gc.set_step("repeat", 1)


# gc.display_stats()
# gc.glitch_plot(plotdots={"success":"+r", "reset":"xy", "normal":'xg'})

In [10]:
gc = cw.GlitchController(groups=["success", "reset", "normal"], parameters=["ext_offset", "repeat"])
min_delay = int(20e-6 * scope.clock.clkgen_freq)
max_delay = int(25e-6 * scope.clock.clkgen_freq)

print(min_delay, max_delay)

gc.set_range("ext_offset", min_delay, max_delay)
gc.set_range("repeat", 7, 12) 
gc.set_global_step(1)

gc.display_stats()
gc.glitch_plot(plotdots={"success":"+r", "reset":"xy", "normal":'xg'})

2000 2500


IntText(value=0, description='success count:', disabled=True)

IntText(value=0, description='reset count:', disabled=True)

IntText(value=0, description='normal count:', disabled=True)

FloatSlider(value=2000.0, continuous_update=False, description='ext_offset setting:', disabled=True, max=2500.…

FloatSlider(value=7.0, continuous_update=False, description='repeat setting:', disabled=True, max=12.0, min=7.…

:DynamicMap   []
   :Overlay
      .Points.I   :Points   [ext_offset,repeat]
      .Points.II  :Points   [ext_offset,repeat]
      .Points.III :Points   [ext_offset,repeat]

In [8]:
def make_pkt(data):
    pkt = b"\x01"
    pkt += struct.pack('!H', len(data))
    pkt += data

    # Checksum
    chk = (256 - sum(pkt[1:])) & 0xff
    pkt += bytes([chk, 0x03])

    return pkt

def as_hex(s):
    return "".join([f"\\x{ord(c):02x}" for c in s])

def read_block(target, block, gc):
    start, end = block

    # print("reading flash", hex(start), hex(end))
    pkt = make_pkt(b'\x15' + struct.pack('>II', start, end))

    target.write(pkt)   
    resp = target.read(6)

    if resp == '\x81\x00\x02\x95\xc3\xa6':
        print("Auth error")
        gc.add("normal")
        return None
        
    if resp != '\x81\x00\x01\x15\xea\x03':
        print("wrong read ack", as_hex(resp))
        gc.add("reset")
        return None

    rem = end - start
    dat = b""
    
    while rem > 0:
        target.write(b'\x81\x00\x01\x15\xea\x03')
        length = target.read(3)
        
        if len(length) != 3:
            print("wrong length received", as_hex(length))
            return None
        
        length = bytes([ord(c) for c in length])
        length = struct.unpack('!H', length[1:])[0] - 1 # Data starts with 0x15

        resp = b""
        while len(resp) < length + 3: 
            resp += bytes([ord(c) for c in target.read(target.in_waiting())])
            time.sleep(0.05)

        dat += resp[1:-2]
        rem -= length

    gc.add("success")
    return dat




In [ ]:
blocks = [(0x0, 0x1fff)]

while True:

    for delay, width in gc.glitch_values(clear=False):
        scope.sc.arm(False)
        scope.glitch.ext_offset = delay
        scope.glitch.repeat = width
        scope.glitch.width = 0
        scope.glitch.offset = 0
    
        scope.arm()
        # scope.glitch.manual_trigger()
        
        scope.io.tio3 = 'gpio_high' # Release Reset
        time.sleep(.1)
        target.read(target.in_waiting()) # Clean all bytes in serial input buffer
    
        # print(scope.UARTTrigger.matched_pattern_counts)
        try:
            # Sync
            target.write(b'\x00' * 10)
            assert target.read(1) == '\x00', "Connect error 1"
    
            # Connect
            target.write(b'\x55')
            assert target.read(1) == '\xc1', "Connect error 2"
    
            # Get device type
            target.write(b'\x01\x00\x01\x38\xc7\x03')
            assert target.read(6) == '\x81\x00\x01\x38\xc7\x03', "Wrong Resp Device Type 1"
    
            target.write(b'\x81\x00\x01\x38\xc7\x03')
            resp = target.read(30)
            assert resp == '\x81\x00\x19\x38\x10\x00\xff\x40\x00\x48\x01\x00\x01\x6e\x36\x00\x00\x7a\x12\x00\x05\xb8\xd8\x00\x01\x7d\x78\x40\x1b\x03', "Wrong Resp Device Type 2"
    
            # Set up frequencies
            target.write(make_pkt(b'\x32\x00\xf4\x24\x00\x01\x7d\x78\x40'))
    
            resp = target.read(6)
            assert resp == '\x81\x00\x01\x32\xcd\x03', "wrong resp freq 1"
    
            target.write(b'\x81\x00\x01\x32\xcd\x03')
            resp = target.read(14)
            assert resp == '\x81\x00\x09\x32\x01\x7d\x78\x40\x00\xbe\xbc\x20\xf5\x03', "unexpected resp freq 2"
    
            # Set bitrate (9600 baud)
            target.write(b'\x01\x00\x05\x34\x00\x00\x25\x80\x22\x03')
            resp = target.read(6)
            assert resp == '\x81\x00\x01\x34\xcb\x03', "wrong resp bitrate 1"
    
            # Synchronize!
            target.write(b'\x01\x00\x01\x00\xff\x03')
            resp = target.read(6)
            assert resp == '\x81\x00\x01\x00\xff\x03', "wrong resp synchronize"

            target.write(make_pkt(b'\x30' + (b'\xff' * 16)))
            resp = target.read(7)
            # print(as_hex(resp))

            if resp == '\x81\x00\x02\x30\x00\xce\x03':
                print("OKKK!", delay, width)
                pass
            elif resp == '\x81\x00\x02\xb0\xdb\x73\x03': # Wrong ID
                gc.add("normal")
                continue
            elif resp == '':
                gc.add("reset")
                continue
            elif resp == '\x00':
                gc.add("reset")
                continue
            else:
                print(delay, width, "wrong resp id", as_hex(resp))
                gc.add("success")
                continue
                
            resp = read_block(target, blocks[0], gc)  
            if resp is not None:
                print("Delay", delay)
                print("Width", width)
                print("Got", len(resp), resp.hex())
                raise RuntimeError("DONE!")
    
        except AssertionError as e:
            print("assert", delay, width, e)
            gc.add("reset")
    
        finally:
            scope.io.tio3 = 'gpio_low' # Assert reset
            time.sleep(.1)